# DEDALUS · Perception · U-Net per le linee della corsia — v2

Notebook per Kaggle. Allena una **U-Net** che dice, pixel per pixel, "qui c'è una linea della corsia" oppure no.
Le etichette arrivano gratis dalla segmentazione di CARLA (classe `RoadLine`).

### Cosa c'è di nuovo rispetto alla v1 (e perché)
L'idea guida viene da **nnU-Net** (Isensee et al., *Nature Methods* 2021) e da **nnU-Net Revisited** (MICCAI 2024):
la maggior parte del guadagno non viene da un'architettura "più nuova" (Transformer, Mamba), ma da una **U-Net CNN
configurata bene**: encoder residuo, loss Dice + Cross-Entropy, augmentation e schedule del learning rate fatti con cura.
Quindi qui **non** cambiamo tipo di rete: miglioriamo la ricetta.

| | v1 | v2 |
|---|---|---|
| Modelli | U-Net del lab 9, U-Net con ResNet-18 | + **U-Net residua** da zero (encoder a blocchi residui, idea "ResEnc" di nnU-Net), 4 livelli invece di 2 |
| Loss | Cross-Entropy pesata | **Cross-Entropy pesata + Dice** (la combinazione di nnU-Net) + opzionale **clDice** per strutture sottili |
| Augmentation | ColorJitter + flip | + piccola **affine** (rotazione, spostamento, zoom) e **blur**, applicate insieme a immagine e maschera con `tv_tensors` |
| Learning rate | fisso | **warm-up + cosine**, weight decay, gradient clipping |
| Velocità | PyTorch FP32 | **FP16** in inferenza, `channels_last`, **export ONNX** + misura FPS con ONNX Runtime (il formato che serve per il Jetson) |

**Da dove viene il codice "base"**: lab 9 *Semantic Segmentation* (AN2DL 2025-26) per `UNetBlock`, `UNet`, Dataset e ciclo
di allenamento; lab 7 *Transfer Learning and Fine-Tuning* (AN2DL 2026-27) per l'encoder pre-allenato. Le fonti delle parti nuove
sono citate nelle celle dove compaiono e raccolte alla fine.

**Prima di eseguire, su Kaggle:**
1. *Add Input* → carica la cartella con le sessioni (es. `dedalus-lane/s01/rgb/...`).
2. *Settings → Accelerator* → **GPU T4 x2** (ne usiamo una) o **P100**.
3. *Settings → Internet* → **On** (pacchetti e pesi ImageNet).

Struttura dei dati attesa (una cartella per sessione):
```
<DATA_ROOT>/
  s01/  rgb/000000.png   segmentazione/000000.png   profondita/000000.npy   lidar/000000.npy
  s02/  ...
```
Parti: **(1) dati → (2) modelli → (3) allenamento → (4) valutazione → (5) ottimizzazione per il robot (ONNX)**.

In [ ]:
# torchmetrics: metriche dei lab | onnx, onnxscript, onnxruntime: export e test di velocità (parte 5)
!pip install -q torchmetrics onnx onnxscript onnxruntime-gpu

## 0. Impostazioni
Tutto quello che potreste voler cambiare sta in questa cella.

In [ ]:
import os, glob, json, time, random, math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2 as transforms
import torchmetrics
from PIL import Image
import matplotlib.pyplot as plt

# --- Dove sono i dati -------------------------------------------------------
DATA_ROOT = "/kaggle/input"          # il notebook cerca le sessioni qui dentro, anche in sottocartelle
CARTELLA_RGB = "rgb"
CARTELLA_SEG = "segmentazione"

# --- Etichette --------------------------------------------------------------
ROADLINE_ID = 24     # id della classe RoadLine in CARLA 0.9.14+ (controllato nella cella 1.2)
NUM_CLASSES = 2      # 0 = sfondo, 1 = linea della corsia

# --- Divisione train / val / test PER SESSIONE ------------------------------
SPLIT = {
    "train": [],   # es. ["s01", "s02", "s04"]
    "val":   [],   # es. ["s03"]
    "test":  [],   # es. ["s05"]
}

# --- Modello ----------------------------------------------------------------
# "lab"          = U-Net del lab 9 (2 livelli): il riferimento di partenza
# "res_unet"     = U-Net residua da zero, 4 livelli (idea ResEnc di nnU-Net)
# "pre_allenato" = encoder ResNet-18 ImageNet + decoder UNetBlock (lab 7): di solito il migliore con pochi dati
MODELLO = "pre_allenato"

# --- Loss -------------------------------------------------------------------
PESO_DICE = 1.0       # Cross-Entropy + Dice, come nnU-Net
PESO_CLDICE = 0.0     # clDice per linee sottili: provate 0.5 dopo il primo confronto

# --- Immagini e allenamento -------------------------------------------------
INPUT_SIZE = (288, 512)     # (altezza, larghezza); multipli di 32
BATCH_SIZE = 8              # se compare "CUDA out of memory" (soprattutto con res_unet), scendete a 4
EPOCHS = 150
EPOCHE_WARMUP = 5           # il learning rate sale piano nelle prime epoche, poi scende a coseno
LEARNING_RATE = 3e-4 if MODELLO == "pre_allenato" else 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 25               # early stopping sulla IoU di validazione
NUM_WORKERS = 2
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True     # immagini sempre della stessa dimensione: cuDNN sceglie l'algoritmo più veloce
else:
    device = torch.device("cpu")
USA_AMP = device.type == "cuda"               # mixed precision (FP16) solo su GPU
print(f"PyTorch {torch.__version__} | device: {device} | modello: {MODELLO}")

# Parte 1 · Dati ed etichette

### 1.1 Trovare le sessioni
Una cartella è una "sessione" se contiene sia `rgb/` sia `segmentazione/`.

In [ ]:
def trova_sessioni(root):
    sessioni = {}
    for rgb_dir in glob.glob(os.path.join(root, "**", CARTELLA_RGB), recursive=True):
        cartella = os.path.dirname(rgb_dir)
        if os.path.isdir(os.path.join(cartella, CARTELLA_SEG)):
            sessioni[os.path.basename(cartella)] = cartella
    return dict(sorted(sessioni.items()))

def frame_di_sessione(cartella):
    coppie = []
    for rgb in sorted(glob.glob(os.path.join(cartella, CARTELLA_RGB, "*.png"))):
        seg = os.path.join(cartella, CARTELLA_SEG, os.path.basename(rgb))
        if os.path.exists(seg):
            coppie.append((rgb, seg))
    return coppie

SESSIONI = trova_sessioni(DATA_ROOT)
for nome, cartella in SESSIONI.items():
    print(f"{nome}: {len(frame_di_sessione(cartella))} frame   ({cartella})")
assert SESSIONI, "Nessuna sessione trovata: controlla DATA_ROOT e i nomi delle cartelle."

### 1.2 Dalla segmentazione di CARLA alla maschera "linea / non linea"
Come la `category_map` del lab 9, in versione estrema: tutte le classi diventano 0, tranne `RoadLine` che diventa 1.
Nella PNG di segmentazione di CARLA il **canale rosso** contiene il numero della classe.

In [ ]:
def carica_maschera(percorso_seg):
    seg = np.array(Image.open(percorso_seg).convert("RGB"))
    return (seg[:, :, 0] == ROADLINE_ID).astype(np.int64)

esempio_rgb, esempio_seg = frame_di_sessione(next(iter(SESSIONI.values())))[0]
rosso = np.array(Image.open(esempio_seg).convert("RGB"))[:, :, 0]
ids, conteggi = np.unique(rosso, return_counts=True)
print("Classi presenti (id: % pixel):")
for i, c in zip(ids, conteggi):
    print(f"  {i:3d}: {100*c/rosso.size:5.2f} %")

maschera = carica_maschera(esempio_seg)
print(f"\nPixel di linea nel frame: {maschera.mean()*100:.2f} %")
if maschera.sum() == 0:
    print("ATTENZIONE: nessun pixel RoadLine. Controlla ROADLINE_ID (nelle versioni vecchie di CARLA era 6).")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].imshow(Image.open(esempio_rgb)); ax[0].set_title("RGB")
ax[1].imshow(maschera, cmap="gray"); ax[1].set_title("Etichetta: linee della corsia")
for a in ax: a.axis("off")
plt.show()

### 1.3 Divisione train / val / test (per sessione)
I frame vengono da un **video**: frame vicini sono quasi uguali. Se uno finisce nel train e il "gemello" nel test,
l'IoU sembra più alta del vero. Per questo si divide **per sessione**, mai per frame.
Con **una sola sessione** il notebook la taglia in blocchi di tempo con un buco in mezzo: va bene per provare il codice,
ma i numeri saranno ottimisti. Appena avete più sessioni, riempite `SPLIT`.

In [ ]:
def dividi_dati():
    if any(SPLIT.values()):
        return {parte: [f for s in nomi for f in frame_di_sessione(SESSIONI[s])]
                for parte, nomi in SPLIT.items()}
    if len(SESSIONI) > 1:
        raise ValueError("Ci sono più sessioni: scrivete quali vanno in train/val/test in SPLIT (cella 0).")
    print("Una sola sessione: divisione TEMPORALE provvisoria (70% train, 15% val, 15% test, con buchi).")
    frame = frame_di_sessione(next(iter(SESSIONI.values())))
    n = len(frame)
    buco = max(1, n // 25)
    fine_train, fine_val = int(0.70 * n), int(0.85 * n)
    return {"train": frame[:fine_train - buco], "val": frame[fine_train:fine_val - buco], "test": frame[fine_val:]}

DATI = dividi_dati()
for parte, lista in DATI.items():
    print(f"{parte:5s}: {len(lista)} frame")

### 1.4 Dataset e augmentation
Struttura del `SegmentationDataset` del lab 9, con una novità: le augmentation sono fatte con i **`tv_tensors`** di
torchvision v2. Si "etichetta" l'immagine come `tv_tensors.Image` e la maschera come `tv_tensors.Mask`, e **una sola
chiamata** applica a tutte e due la stessa trasformazione geometrica (con interpolazione *nearest* sulla maschera, che resta 0/1).
Le trasformazioni di colore toccano solo l'immagine.

Augmentation (tutte piccole: la camera è fissa sull'auto, non vogliamo immagini impossibili):
- specchio orizzontale;
- **affine**: rotazione ±3°, spostamento ±3%, zoom 0.9–1.1 (simula sobbalzi, beccheggio e piccole differenze di montaggio);
- **ColorJitter** (luce, contrasto, saturazione) e **blur** (simula movimento e messa a fuoco).

Normalizzazione ImageNet solo per il modello pre-allenato, come nel lab 7.

In [ ]:
MEDIA_IMAGENET = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
DEV_STD_IMAGENET = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def carica_rgb(img_pil):
    img = img_pil.convert("RGB").resize((INPUT_SIZE[1], INPUT_SIZE[0]), Image.BILINEAR)
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)

def normalizza(x):
    return (x - MEDIA_IMAGENET) / DEV_STD_IMAGENET if MODELLO == "pre_allenato" else x

def prepara_rgb(img_pil):
    return normalizza(carica_rgb(img_pil))

class LaneDataset(Dataset):
    def __init__(self, coppie, augmentation=None, maschera_originale=False):
        self.coppie = coppie
        self.augmentation = augmentation
        self.maschera_originale = maschera_originale

    def __len__(self):
        return len(self.coppie)

    def __getitem__(self, idx):
        percorso_rgb, percorso_seg = self.coppie[idx]
        image = tv_tensors.Image(carica_rgb(Image.open(percorso_rgb)))
        mask_full = carica_maschera(percorso_seg)
        small = Image.fromarray(mask_full.astype(np.uint8)).resize((INPUT_SIZE[1], INPUT_SIZE[0]), Image.NEAREST)
        label = tv_tensors.Mask(torch.from_numpy(np.array(small)).long())

        if self.augmentation is not None:
            image, label = self.augmentation(image, label)     # stessa geometria per immagine e maschera

        image = normalizza(image.as_subclass(torch.Tensor))
        label = label.as_subclass(torch.Tensor)
        if self.maschera_originale:
            return image, label, torch.from_numpy(mask_full)
        return image, label

train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=3, translate=(0.03, 0.03), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.2),
])

train_dataset = LaneDataset(DATI["train"], augmentation=train_augmentation)
val_dataset   = LaneDataset(DATI["val"])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=len(train_dataset) > BATCH_SIZE, persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

x, y = next(iter(train_loader))
print("Batch immagini:", tuple(x.shape), " batch maschere:", tuple(y.shape))
print(f"Pixel di linea nel batch: {y.float().mean().item()*100:.2f} %  -> classe molto sbilanciata")

def per_mostrare(img):
    if MODELLO == "pre_allenato":
        img = img * DEV_STD_IMAGENET + MEDIA_IMAGENET
    return img.permute(1, 2, 0).clamp(0, 1).numpy()

fig, ax = plt.subplots(2, 4, figsize=(16, 5))
for k in range(min(4, len(x))):
    ax[0, k].imshow(per_mostrare(x[k]))
    ax[1, k].imshow(y[k], cmap="gray")
for a in ax.ravel(): a.axis("off")
plt.suptitle("Esempi di train dopo augmentation (controllate che linee e maschere coincidano)"); plt.show()

# Parte 2 · I modelli

### 2.1 U-Net del lab 9 (il riferimento)
Identica al lab, con `num_classes=2`. Ha solo **2 livelli**: vede un contesto piccolo intorno a ogni pixel.
La teniamo come riferimento: ogni miglioramento va misurato contro di lei.

In [ ]:
class UNetBlock(nn.Module):
    # Conv-BN-ReLU ripetuto (identico al lab 9)
    def __init__(self, in_channels, out_channels, kernel_size=3, num_convs=2):
        super().__init__()
        layers = []
        for i in range(num_convs):
            layers.extend([
                nn.Conv2d(in_channels if i == 0 else out_channels, out_channels,
                          kernel_size=kernel_size, padding=kernel_size // 2),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            ])
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    # U-Net del lab 9
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.down_block1 = UNetBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.down_block2 = UNetBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.bottleneck = UNetBlock(64, 128)
        self.up1 = nn.Upsample(scale_factor=2, mode="nearest")
        self.up_block1 = UNetBlock(128 + 64, 64)
        self.up2 = nn.Upsample(scale_factor=2, mode="nearest")
        self.up_block2 = UNetBlock(64 + 32, 32)
        self.output = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        d1 = self.down_block1(x);  p1 = self.pool1(d1)
        d2 = self.down_block2(p1); p2 = self.pool2(d2)
        b = self.bottleneck(p2)
        u1 = self.up_block1(torch.cat([self.up1(b), d2], dim=1))
        u2 = self.up_block2(torch.cat([self.up2(u1), d1], dim=1))
        return self.output(u2)

### 2.2 U-Net residua (da zero)
Due cambiamenti rispetto al lab, entrambi "classici" ma efficaci:
- **blocchi residui** (He et al., ResNet): l'uscita del blocco è `blocco(x) + x`. Il gradiente passa più facilmente,
  quindi si può fare una rete più profonda senza che l'allenamento si blocchi. nnU-Net Revisited mostra che un encoder
  residuo ("ResEnc") è la scelta più solida per le U-Net;
- **4 livelli** invece di 2: ogni pixel "vede" una zona molto più grande dell'immagine. Serve per le linee tratteggiate,
  dove la rete deve capire che due pezzi lontani fanno parte della stessa linea.

Il rimpicciolimento si fa con una convoluzione con `stride=2` (impara *come* rimpicciolire) invece del `MaxPool`.
Nel decoder usiamo `Upsample` **bilineare**, più morbido del *nearest*.

In [ ]:
class ResBlock(nn.Module):
    # Conv-BN-ReLU-Conv-BN + scorciatoia (residuo), poi ReLU
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.scorciatoia = nn.Identity()
        if stride != 1 or in_channels != out_channels:   # se cambia forma, adattiamo x con una conv 1x1
            self.scorciatoia = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                                             nn.BatchNorm2d(out_channels))

    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return self.relu(y + self.scorciatoia(x))


class ResUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2, canali=(32, 64, 128, 256, 320)):
        super().__init__()
        # Encoder: a ogni livello 2 blocchi residui; il primo dimezza la risoluzione (stride 2)
        self.encoder = nn.ModuleList()
        c_prec = in_channels
        for i, c in enumerate(canali):
            self.encoder.append(nn.Sequential(ResBlock(c_prec, c, stride=1 if i == 0 else 2), ResBlock(c, c)))
            c_prec = c
        # Decoder: Upsample bilineare -> concatena con l'encoder -> UNetBlock (lab 9)
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.decoder = nn.ModuleList()
        for c_basso, c_skip in zip(canali[::-1][:-1], canali[::-1][1:]):
            self.decoder.append(UNetBlock(c_basso + c_skip, c_skip))
            canali_finali = c_skip
        self.output = nn.Conv2d(canali_finali, num_classes, kernel_size=1)

    def forward(self, x):
        salti = []
        for livello in self.encoder:
            x = livello(x)
            salti.append(x)
        x = salti.pop()
        for blocco in self.decoder:
            x = blocco(torch.cat([self.up(x), salti.pop()], dim=1))
        return self.output(x)

### 2.3 U-Net con encoder pre-allenato (lab 7)
Encoder = ResNet-18 di `torchvision` con pesi ImageNet (sa già riconoscere bordi, colori e texture: con pochi dati è un vantaggio enorme).
Decoder = `UNetBlock` del lab 9 con `Upsample` bilineare. Le uscite della ResNet hanno risoluzione 1/2, 1/4, 1/8, 1/16, 1/32,
quindi l'ingresso deve avere lati **multipli di 32**.

In [ ]:
class UNetPreAllenata(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        r = torchvision.models.resnet18(weights=weights)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)      #  64 canali, 1/2
        self.pool = r.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec4 = UNetBlock(512 + 256, 256)
        self.dec3 = UNetBlock(256 + 128, 128)
        self.dec2 = UNetBlock(128 + 64, 64)
        self.dec1 = UNetBlock(64 + 64, 32)
        self.output = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        s = self.stem(x)
        e1 = self.enc1(self.pool(s))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d = self.dec4(torch.cat([self.up(e4), e3], dim=1))
        d = self.dec3(torch.cat([self.up(d), e2], dim=1))
        d = self.dec2(torch.cat([self.up(d), e1], dim=1))
        d = self.dec1(torch.cat([self.up(d), s], dim=1))
        return self.output(self.up(d))

In [ ]:
def crea_modello(nome=MODELLO, pretrained=True):
    if nome == "lab":
        return UNet(in_channels=3, num_classes=NUM_CLASSES)
    if nome == "res_unet":
        return ResUNet(in_channels=3, num_classes=NUM_CLASSES)
    return UNetPreAllenata(NUM_CLASSES, pretrained=pretrained)

# channels_last: su GPU NVIDIA le convoluzioni (soprattutto in FP16) sono più veloci con questa disposizione in memoria
model = crea_modello().to(device, memory_format=torch.channels_last)
n_param = sum(p.numel() for p in model.parameters())
print(f"Modello: {MODELLO} | parametri: {n_param/1e6:.2f} milioni")

with torch.no_grad():
    uscita = model(x[:1].to(device, memory_format=torch.channels_last))
print("Ingresso:", tuple(x[:1].shape), "-> uscita:", tuple(uscita.shape))

# Parte 3 · Allenamento

### 3.1 La loss: Cross-Entropy pesata + Dice (+ clDice opzionale)
- **Cross-Entropy pesata** (come nel lab, con `weight=`): giudica ogni pixel; il peso alza l'importanza della classe linea,
  che è solo l'1–3 % dei pixel. Con il Dice accanto basta un peso moderato (radice del rapporto sfondo/linea).
- **Dice loss** (Milletari et al., V-Net): misura quanto si sovrappongono maschera predetta e vera, quindi non si fa
  ingannare dallo sbilanciamento. **Cross-Entropy + Dice è la loss di default di nnU-Net.**
- **clDice** (Shit et al., CVPR 2021), opzionale con `PESO_CLDICE`: confronta gli **scheletri** (le linee centrali)
  delle maschere. È pensata per strutture sottili e allungate (vasi, strade, crepe): premia la rete se non "spezza" le linee.
  È una prova da fare, non una certezza: le nostre linee tratteggiate sono interrotte di proposito.

Metriche come prima: **IoU della classe linea** con `torchmetrics.JaccardIndex(task="binary")`, e precision / recall / F1
dagli stessi conteggi TP, FP, FN, per il confronto con la pipeline di Ale.

In [ ]:
pixel_linea = sum(LaneDataset([c])[0][1].float().mean().item() for c in DATI["train"]) / len(DATI["train"])
peso_linea = float(np.clip(math.sqrt((1 - pixel_linea) / max(pixel_linea, 1e-6)), 1.0, 20.0))
CLASS_WEIGHTS = torch.tensor([1.0, peso_linea], device=device)
print(f"Pixel di linea nel train: {pixel_linea*100:.2f} %  ->  peso della classe linea nella CE: {peso_linea:.1f}")
criterion_ce = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

def dice_loss(prob, target, eps=1.0):
    # "batch Dice": somme su tutto il batch, più stabile quando un'immagine ha poche linee
    inter = (prob * target).sum()
    return 1 - (2 * inter + eps) / (prob.sum() + target.sum() + eps)

# --- clDice (Shit et al. 2021): scheletro "morbido" fatto con max-pooling, quindi derivabile ---
def soft_erode(img):
    p1 = -F.max_pool2d(-img, (3, 1), 1, (1, 0))
    p2 = -F.max_pool2d(-img, (1, 3), 1, (0, 1))
    return torch.min(p1, p2)

def soft_dilate(img):
    return F.max_pool2d(img, 3, 1, 1)

def soft_skel(img, iters=10):
    skel = F.relu(img - soft_dilate(soft_erode(img)))
    for _ in range(iters):
        img = soft_erode(img)
        delta = F.relu(img - soft_dilate(soft_erode(img)))
        skel = skel + F.relu(delta - skel * delta)
    return skel

def cldice_loss(prob, target, eps=1.0):
    prob, target = prob[:, None], target[:, None]
    skel_p, skel_t = soft_skel(prob), soft_skel(target)
    t_prec = ((skel_p * target).sum() + eps) / (skel_p.sum() + eps)
    t_sens = ((skel_t * prob).sum() + eps) / (skel_t.sum() + eps)
    return 1 - 2 * t_prec * t_sens / (t_prec + t_sens)

def loss_fn(logits, labels):
    logits = logits.float()                         # la loss sempre in FP32, anche con la mixed precision
    loss = criterion_ce(logits, labels)
    prob = torch.softmax(logits, dim=1)[:, 1]
    target = labels.float()
    if PESO_DICE > 0:
        loss = loss + PESO_DICE * dice_loss(prob, target)
    if PESO_CLDICE > 0:
        loss = loss + PESO_CLDICE * cldice_loss(prob, target)
    return loss

def conta(pred, target):
    pred, target = pred.bool(), target.bool()
    return (pred & target).sum().item(), (pred & ~target).sum().item(), (~pred & target).sum().item()

def metriche(tp, fp, fn):
    eps = 1e-9
    iou = tp / (tp + fp + fn + eps)
    p = tp / (tp + fp + eps)
    r = tp / (tp + fn + eps)
    return {"IoU": iou, "Precision": p, "Recall": r, "F1": 2 * p * r / (p + r + eps)}

### 3.2 Ciclo di allenamento
Stessa struttura di `train_segmentation_model` del lab 9, con quattro aggiunte:
- **AdamW con weight decay** (Loshchilov & Hutter, ICLR 2019): il "decadimento" dei pesi fatto nel modo corretto per Adam;
- **warm-up + cosine**: il learning rate parte basso per `EPOCHE_WARMUP` epoche (i primi passi sono i più instabili,
  soprattutto con pesi pre-allenati), poi scende dolcemente a forma di coseno fino a quasi zero;
- **gradient clipping**: se un batch produce un gradiente enorme, lo si "taglia" (nnU-Net usa la soglia 12);
- **mixed precision** (FP16 con `GradScaler`), già nel lab: più veloce e meno memoria su GPU.

In [ ]:
def train_segmentation_model(model, train_loader, val_loader, epochs, learning_rate, patience, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=EPOCHE_WARMUP)
    coseno = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs - EPOCHE_WARMUP), eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, coseno], milestones=[EPOCHE_WARMUP])
    scaler = torch.amp.GradScaler(enabled=USA_AMP)
    val_iou_metric = torchmetrics.JaccardIndex(task="binary").to(device)

    history = {"loss": [], "val_loss": [], "val_iou": [], "lr": []}
    best_val_iou, patience_counter, best_epoch, best_state = -1.0, 0, 0, None

    for epoch in range(epochs):
        # ---------- TRAINING ----------
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=USA_AMP):
                logits = model(images)
            loss = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=12.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * images.size(0)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        scheduler.step()

        # ---------- VALIDAZIONE ----------
        model.eval()
        val_loss = 0.0
        val_iou_metric.reset()
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, memory_format=torch.channels_last)
                labels = labels.to(device)
                with torch.amp.autocast(device_type=device.type, enabled=USA_AMP):
                    logits = model(images)
                val_loss += loss_fn(logits, labels).item() * images.size(0)
                val_iou_metric.update(logits.argmax(dim=1), labels)
        val_iou = val_iou_metric.compute().item()

        history["loss"].append(train_loss / len(train_loader.dataset))
        history["val_loss"].append(val_loss / len(val_loader.dataset))
        history["val_iou"].append(val_iou)
        print(f"Epoch {epoch+1:3d}/{epochs} | lr {history['lr'][-1]:.1e} | Loss: {history['loss'][-1]:.4f} | "
              f"Val Loss: {history['val_loss'][-1]:.4f} | Val IoU (linea): {val_iou:.4f}")

        # ---------- EARLY STOPPING ----------
        if val_iou > best_val_iou:
            best_val_iou, best_epoch, patience_counter = val_iou, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping all'epoca {epoch+1}")
                break

    model.load_state_dict(best_state)
    print(f"Ripristinato il modello migliore: epoca {best_epoch+1}, Val IoU {best_val_iou:.4f}")
    return model, history

%time model, history = train_segmentation_model(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, PATIENCE, device)
NOME_PESI = f"unet_lane_{MODELLO}.pt"
torch.save(model.state_dict(), NOME_PESI)
print("Pesi salvati in", NOME_PESI)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(history["loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss (CE + Dice)"); ax[0].set_xlabel("epoca"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history["val_iou"]); ax[1].set_title("IoU di validazione (classe linea)"); ax[1].set_xlabel("epoca"); ax[1].grid(alpha=0.3)
ax[2].plot(history["lr"]); ax[2].set_title("Learning rate (warm-up + coseno)"); ax[2].set_xlabel("epoca"); ax[2].grid(alpha=0.3)
plt.show()

# Parte 4 · Valutazione

### 4.1 Soglia scelta sulla validazione
"Linea" quando la probabilità della classe linea supera la **soglia**. Proviamo alcune soglie **sulla validazione**;
il test non si tocca per queste scelte.

In [ ]:
model.eval()
SOGLIE = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
conteggi = {s: [0, 0, 0] for s in SOGLIE}
with torch.no_grad():
    for images, labels in val_loader:
        with torch.amp.autocast(device_type=device.type, enabled=USA_AMP):
            logits = model(images.to(device, memory_format=torch.channels_last))
        prob_linea = torch.softmax(logits.float(), dim=1)[:, 1]
        for s in SOGLIE:
            a, b, c = conta(prob_linea > s, labels.to(device))
            conteggi[s][0] += a; conteggi[s][1] += b; conteggi[s][2] += c

for s in SOGLIE:
    print(f"soglia {s:.1f}: val IoU {metriche(*conteggi[s])['IoU']:.3f}")
SOGLIA = max(SOGLIE, key=lambda s: metriche(*conteggi[s])["IoU"])
print("Soglia scelta:", SOGLIA)

### 4.2 Risultati sul test (stesso protocollo della pipeline di Ale)
Maschera riportata a **960×540** e confrontata con quella originale; tempo misurato su un frame alla volta (batch 1),
lettura da disco esclusa, riscaldamento escluso. In inferenza su GPU usiamo **FP16** (`autocast`): stessa precisione
in pratica, tempo più basso.

In [ ]:
parte_test = "test" if DATI["test"] else "val"
test_dataset = LaneDataset(DATI[parte_test], maschera_originale=True)
print(f"Valuto su '{parte_test}': {len(test_dataset)} frame")

def predici_maschera(img_pil):
    # Un frame RGB (PIL, 960x540) -> maschera 0/1 alla risoluzione originale
    W, H = img_pil.size
    x = prepara_rgb(img_pil)[None].to(device, memory_format=torch.channels_last)
    with torch.no_grad(), torch.amp.autocast(device_type=device.type, enabled=USA_AMP):
        logit = model(x)
    logit = F.interpolate(logit.float(), size=(H, W), mode="bilinear", align_corners=False)
    return (torch.softmax(logit, dim=1)[0, 1] > SOGLIA).to("cpu", torch.uint8)

img0 = Image.open(test_dataset.coppie[0][0]).convert("RGB")
for _ in range(10):
    predici_maschera(img0)

tp = fp = fn = 0
tempi = []
for percorso_rgb, percorso_seg in test_dataset.coppie:
    img = Image.open(percorso_rgb).convert("RGB")
    gt = torch.from_numpy(carica_maschera(percorso_seg))
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    pred = predici_maschera(img)
    if device.type == "cuda": torch.cuda.synchronize()
    tempi.append(time.perf_counter() - t0)
    a, b, c = conta(pred, gt)
    tp, fp, fn = tp + a, fp + b, fn + c

risultati = metriche(tp, fp, fn)
risultati["ms_per_frame"] = 1000 * float(np.median(tempi))
risultati["FPS"] = 1000 / risultati["ms_per_frame"]
risultati.update({"split": parte_test, "n_frame": len(tempi), "soglia": SOGLIA, "modello": MODELLO,
                  "peso_dice": PESO_DICE, "peso_cldice": PESO_CLDICE, "parametri_milioni": round(n_param / 1e6, 2),
                  "risoluzione_rete": f"{INPUT_SIZE[1]}x{INPUT_SIZE[0]}", "device": str(device), "fp16": USA_AMP})

print(f"\n=== U-Net ({MODELLO}), risultati ===")
for k, v in risultati.items():
    print(f"{k:18s}: {v:.3f}" if isinstance(v, float) else f"{k:18s}: {v}")
with open(f"risultati_unet_{MODELLO}.json", "w") as f:
    json.dump(risultati, f, indent=2)

### 4.3 Guardare gli errori
Verde = linea trovata (TP) · Rosso = falso allarme (FP) · Blu = linea mancata (FN).

In [ ]:
def mostra_errori(coppia):
    img = Image.open(coppia[0]).convert("RGB")
    gt = carica_maschera(coppia[1]).astype(bool)
    pred = predici_maschera(img).numpy().astype(bool)
    vista = np.asarray(img).copy()
    vista[pred & gt] = [0, 255, 0]
    vista[pred & ~gt] = [255, 0, 0]
    vista[~pred & gt] = [0, 0, 255]
    return vista

campioni = test_dataset.coppie[:: max(1, len(test_dataset) // 3)][:3]
fig, ax = plt.subplots(len(campioni), 1, figsize=(12, 4 * len(campioni)), squeeze=False)
for k, c in enumerate(campioni):
    ax[k, 0].imshow(mostra_errori(c)); ax[k, 0].set_title(os.path.basename(c[0])); ax[k, 0].axis("off")
plt.tight_layout(); plt.show()

# Parte 5 · Ottimizzazione per il robot: export ONNX

Sul Jetson del robot non useremo PyTorch ma un motore più veloce (TensorRT). Il passo comune è il formato **ONNX**:
un unico file `.onnx` con rete e pesi, che si apre con **ONNX Runtime** (anche su PC) e si converte in TensorRT sul Jetson.

Qui facciamo tre cose:
1. esportiamo il modello con `torch.onnx.export(..., dynamo=True)` (il metodo raccomandato da PyTorch dalla 2.5; se fallisce
   si usa quello classico);
2. controlliamo che ONNX dia **gli stessi numeri** di PyTorch;
3. misuriamo i **ms per frame** con ONNX Runtime. Sono questi i numeri da confrontare con YOLO (stesso procedimento).

In [ ]:
import onnxruntime as ort

model.eval()
modello_export = crea_modello(pretrained=False).to("cpu")   # copia in FP32 su CPU (i pesi li copiamo sotto)
modello_export.load_state_dict({k: v.to("cpu") for k, v in model.state_dict().items()})
modello_export.eval()
ingresso = torch.randn(1, 3, *INPUT_SIZE)
FILE_ONNX = f"unet_lane_{MODELLO}.onnx"

try:
    torch.onnx.export(modello_export, (ingresso,), FILE_ONNX, dynamo=True, external_data=False,
                      input_names=["image"], output_names=["logits"])   # external_data=False: pesi DENTRO il .onnx, un file solo
    print("Esportato con il nuovo exporter (dynamo=True)")
except Exception as e:
    print("dynamo=True non riuscito:", type(e).__name__, "-> uso l'exporter classico")
    torch.onnx.export(modello_export, (ingresso,), FILE_ONNX, dynamo=False, opset_version=17,
                      input_names=["image"], output_names=["logits"])
print(f"{FILE_ONNX}: {os.path.getsize(FILE_ONNX)/1e6:.1f} MB")

provider = [p for p in ["CUDAExecutionProvider", "CPUExecutionProvider"] if p in ort.get_available_providers()]
sessione = ort.InferenceSession(FILE_ONNX, providers=provider)
print("ONNX Runtime usa:", sessione.get_providers()[0])

# 1) stessi numeri di PyTorch?
with torch.no_grad():
    rif = modello_export(ingresso).numpy()
out = sessione.run(None, {sessione.get_inputs()[0].name: ingresso.numpy()})[0]
print(f"Differenza massima PyTorch vs ONNX: {np.abs(rif - out).max():.2e}  (deve essere piccola, < 1e-3)")

# 2) velocità (batch 1, solo la rete)
x_np = prepara_rgb(img0)[None].numpy()
nome_ingresso = sessione.get_inputs()[0].name
for _ in range(10):
    sessione.run(None, {nome_ingresso: x_np})
tempi_onnx = []
for _ in range(50):
    t0 = time.perf_counter(); sessione.run(None, {nome_ingresso: x_np}); tempi_onnx.append(time.perf_counter() - t0)
ms_onnx = 1000 * float(np.median(tempi_onnx))
print(f"ONNX Runtime ({sessione.get_providers()[0]}): {ms_onnx:.1f} ms per frame  ->  {1000/ms_onnx:.0f} FPS (solo rete)")

risultati["onnx_ms_rete"] = ms_onnx
with open(f"risultati_unet_{MODELLO}.json", "w") as f:
    json.dump(risultati, f, indent=2)

## Come usare questo notebook (il confronto da fare)
1. Eseguitelo tre volte, con `MODELLO = "lab"`, `"res_unet"`, `"pre_allenato"`. Il modello "lab" è il riferimento:
   dice quanto valgono davvero le modifiche (è il consiglio principale di *nnU-Net Revisited*: ogni novità va misurata
   contro una base fatta bene, nelle stesse condizioni).
2. Sul migliore, provate `PESO_CLDICE = 0.5` e guardate se la **recall** delle linee lontane migliora.
3. Mettete in una tabella i `risultati_unet_*.json`: IoU, precision, recall, F1, ms per frame (PyTorch e ONNX).
   Aggiungete la riga della pipeline classica di Ale, misurata con lo stesso protocollo.
4. Poi la maschera migliore va nella `perspective.py` di Ale (`INTER_NEAREST`) → sliding window → RANSAC → e_y, e_ψ, κ.

**Attenzione:** con una sola sessione i numeri sono ottimisti e le differenze tra modelli poco affidabili.
Il miglioramento più grande, più di qualsiasi modifica alla rete, sarà **avere più sessioni** (altre mappe, pioggia, notte).

## Fonti
- U-Net: Ronneberger, Fischer, Brox, *MICCAI* 2015 — arxiv.org/abs/1505.04597
- nnU-Net: Isensee et al., *Nature Methods* 18, 203–211 (2021) — doi.org/10.1038/s41592-020-01008-z
- nnU-Net Revisited: Isensee et al., *MICCAI* 2024 — arxiv.org/abs/2404.09556
- ResNet: He et al., *CVPR* 2016 — arxiv.org/abs/1512.03385
- Dice loss (V-Net): Milletari, Navab, Ahmadi, *3DV* 2016 — arxiv.org/abs/1606.04797
- clDice: Shit et al., *CVPR* 2021 — arxiv.org/abs/2003.07311
- AdamW: Loshchilov & Hutter, *ICLR* 2019 — arxiv.org/abs/1711.05101
- LaneNet (segmentazione binaria delle linee + fit): Neven et al., *IEEE IV* 2018 — arxiv.org/abs/1802.05591
- torchvision transforms v2 e `tv_tensors`: docs.pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_getting_started.html
- Export ONNX: docs.pytorch.org/docs/stable/onnx.html